# Cross-disciplinary exchange in biological agent-based modelling

This notebook provides the pipeline for:

1. **Keyword search** to collect articles indexed in the Web of Science Core Collection.
2. **Data cleaning** to remove articles with missing metadata, not written in Enlglish, and those published before 1993.
3. **Coarse-grained LLM classification** to categorise articles as biology or not-biology.
4. **Fine-grained LLM classification** to categories articles into biological subfields (ecology, epidemiology, oncology, or other-biology).

---

## 1a. Keyword search for article collection

Here we use the WoS API to download article metadata with the terms "agent based model" or "individual based model" in the article title, abstract, or author keywords. 

In [ ]:
import requests #allows us to talk to the website
import json #to read and write data in JSON format
import os #to work with folders and file paths
from time import sleep #allows us to pause to not exceed API limit
from datetime import datetime #current time for API

# === CONFIGURATION ===
API_KEY = "1234"   # Dummy WoS API key; replace with your own
BASE_URL = "https://api.clarivate.com/api/wos"  # the website
OUTPUT_DIR = os.path.expanduser("PATH") #output directory
QUERY = 'TI = ("agent based model*" OR "individual based model*") OR AB = ("agent based model*" OR "individual based model*") OR AK = ("agent based model*" OR "individual based model*")'
DELAY_SECONDS = 0.34  # ~3 requests/second to not exceed API limit (how long we wait between requests)

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True) #make the output directory OUTPUT_DIR using the os tooolbox if it does not already exist (exist_ok=TRUE means it's ok if it already exists)

headers = {
    "X-ApiKey": API_KEY,
    "Content-Type": "application/json"
} #this tells the website that we have the API key and that we are expecting JSON files (the output of the API).


def fetch_topic_results(query):
    first_record = 1 #count the first record as 1
    total_fetched = 0 #currently haven't fetched any records...
    session = requests.Session() #part of the requests library to start a connection (and keeps it open rather than reopening it each time) with WoS website

    while True: #keep getting the data whilst there's records to fetch
        try: #if there is an error here, jump to the "except" secion
            params = {
                "databaseId": "WOS",
                "usrQuery": query,
                "count": 100,
                "firstRecord": first_record
            } #these are what we need to input into the WoS API, we can collect 100 at a time
            
            #whatever we collect, call it response
            response = session.get(BASE_URL, headers=headers, params=params, timeout=30) #use our session to go to BASE_URL website and send a request (.get) with the headers and params we defined, if no answer in 30 seconds, give up
            #print(response.text) #see the raw json string from the api before we convert it to a readable format
            response.raise_for_status() #if the website says there's an error, stop

            try: #try this, if it fails then don't crash or stop running
                data = response.json() #convert the json file into to something we can read (stored in a python dictionary), e.g., puts entries onto different lines
            except json.JSONDecodeError: #if it does not or cannot return the readable file (not a json file), then we get an error print statement and stop (break) the while true loop.
                print(f"⚠️ Failed to decode JSON at record {first_record}. Stopping.")
                break

            # Stop if API returned a string (e.g., end of records)
            if not isinstance(data, dict): #isinstance checks if "data" is a python dictionary (it won't be if the api returns something other than a json file)
                print(f"⚠️ API returned a string instead of JSON at record {first_record}. Stopping.")
                break

            # Extract actual article records
            records = data.get("Data", {}).get("Records", {}).get("records", {}).get("REC", []) #the path of where the records are stored in the wos api file (.get gets what is inside that part)
            if not records: #if we did not get any articles then print this ans stop the while true loop
                print(f"⚠️ No records returned at page starting {first_record}.")
                break

            # Print batch info
            batch_size = len(records) #number of articles in this batch
            total_fetched += batch_size #we have now started fetching, so let's count them
            
            start = first_record
            end = first_record + batch_size - 1
            #progress print statement
            print(f"✓ [{datetime.now().strftime('%H:%M:%S')}] Retrieved {len(records)} records "
                  f"({first_record}-{first_record+len(records)-1}), total so far: {total_fetched}")

            # Save only the records to JSON
            filename = f"topic_results_{first_record}.json" #name of the json file (f means string)
            with open(os.path.join(OUTPUT_DIR, filename), "w") as f: #opens/overwrites/creates file name "filename" inside "OUTPUT_DIR" for putting data into (writing - "w")
                # for now, call it "f"
                #"with" ensures the file is closed when done
                json.dump(records, f, indent=2) #json.dump takes the "records" article data and writes it to "f" in json format, indent 2 puts each line 2 spaces inwards when things are put on new lines for readability

            # Pagination: stop if no more records returned
            if batch_size < 100: #if the total number of records in this batch is not 100 (therefore we are on the last batchas we collect 100 articles at a time), print and stop
                print("✅ Last batch reached, stopping.")
                break

            first_record += batch_size #move onto the next 100
            sleep(DELAY_SECONDS) #but wait a little not to exceed the API limit

        except requests.exceptions.RequestException as e: #this basically tells us the error if something goes wrong in the try loop instead of crashing the code!
            print(f"✗ Request failed at record {first_record}: {e}") #prints the error we called "e" and stops the while true loop.
            break

    print(f"\n✅ Done. Total records fetched: {total_fetched}")
    print(f"📂 Saved to: {OUTPUT_DIR}")

#START OF THE CODE!
if __name__ == "__main__":
    fetch_topic_results(QUERY)



## 1ab. Extract the data from the downloaded JSON files into tex format for readability

In [ ]:
import json
import os

# Folder containing the JSON files
json_folder = os.path.expanduser("PATH")
output_file = os.path.join(json_folder, "extracted_wos_data.txt")

def get_nested(data, *keys):
    """Safely get a nested key, return None if missing"""
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return None
    return data

records_text = []

for filename in os.listdir(json_folder):
    if not filename.endswith(".json"):
        continue
    
    with open(os.path.join(json_folder, filename), "r", encoding="utf-8") as f:
        data = json.load(f)
    
    recs = get_nested(data, "records", "REC")
    if not recs:
        continue

    for rec in recs:
        static = get_nested(rec, "static_data")
        if not static:
            continue
        
        summary = get_nested(static, "summary")
        if not summary:
            continue

        # ---- Extract fields ----
        uid = rec.get("UID", "")
        pub_info = get_nested(summary, "pub_info")
        year = get_nested(pub_info, "pubyear")

        # Title
        title_entry = get_nested(summary, "titles", "title")
        title = ""
        source = ""
        if isinstance(title_entry, list):
            for t in title_entry:
                if t.get("type") == "item":
                    title = t.get("content", "")
                elif t.get("type") == "source":
                    source = t.get("content", "")
        elif isinstance(title_entry, dict):
            if title_entry.get("type") == "item":
                title = title_entry.get("content", "")
            elif title_entry.get("type") == "source":
                source = title_entry.get("content", "")

        # Authors
        names = get_nested(summary, "names", "name")
        if isinstance(names, list):
            #authors = "\n   ".join([n.get("display_name", "") for n in names])
            authors = " ".join([n.get("display_name", "") for n in names])
        elif isinstance(names, dict):
            authors = names.get("display_name", "")
        else:
            authors = ""

        # Abstract
        # First try summary path
        abstract = get_nested(summary, "abstracts", "abstract", "abstract_text", "p")
        # If missing, try fullrecord_metadata path
        if not abstract:
            abstract = get_nested(static, "fullrecord_metadata", "abstracts", "abstract", "abstract_text", "p") or ""



        # Keywords
        keywords = get_nested(summary, "keywords", "keyword")
        if isinstance(keywords, list):
            keywords = "; ".join(keywords)
        elif isinstance(keywords, str):
            pass
        else:
            keywords = ""

        # DOI
        doi = ""
        identifiers = get_nested(rec, "dynamic_data", "cluster_related", "identifiers", "identifier")
        if isinstance(identifiers, list):
            for item in identifiers:
                if item.get("type") == "doi":
                    doi = item.get("value", "")
                    break

        #number of references        
        nr = get_nested(static, "fullrecord_metadata", "refs", "count") or 0

        #language
        language = get_nested(static, "fullrecord_metadata", "languages", "language", "content") or ""

        #keywords_plus
        keywords_plus_list = []

        # Correct path: static_data -> item -> keywords_plus -> keyword
        keywords_plus_data = get_nested(static, "item", "keywords_plus", "keyword")
        if isinstance(keywords_plus_data, list):
            keywords_plus_list.extend(keywords_plus_data)
        elif isinstance(keywords_plus_data, str):
            keywords_plus_list.append(keywords_plus_data)

        #keywords_plus = "; ".join(keywords_plus_list)
        keywords_plus = "; ".join(map(str, keywords_plus_list))

        #keywords
        keywords_list = []
        keywords_data = get_nested(static, "fullrecord_metadata", "keywords", "keyword")

        if isinstance(keywords_data, list):
            keywords_list = keywords_data
        elif isinstance(keywords_data, str):
            keywords_list = [keywords_data]

        #keywords = "; ".join(keywords_list)
        keywords = "; ".join(map(str, keywords_list))


        # Citation count (TC)
        citation_data = get_nested(rec, "dynamic_data", "citation_related", "tc_list", "silo_tc")
        tc_count = ""

        if isinstance(citation_data, list):
            for silo in citation_data:
                if silo.get("coll_id") == "WOS":
                    tc_count = silo.get("local_count", "")
                    break
        elif isinstance(citation_data, dict) and citation_data.get("coll_id") == "WOS":
            tc_count = citation_data.get("local_count", "")

        # Document type (DT)
        doctype_data = get_nested(summary, "doctypes", "doctype")
        dt = doctype_data if doctype_data else ""

        # Affiliations (C1)
        # Affiliations (C1)
        addresses_list = []
        addresses_data = get_nested(static, "fullrecord_metadata", "addresses", "address_name")

        if isinstance(addresses_data, list):
            for address in addresses_data:
                full_address = get_nested(address, "address_spec", "full_address")
                if full_address:
                    addresses_list.append(full_address)
        elif isinstance(addresses_data, dict):
            full_address = get_nested(addresses_data, "address_spec", "full_address")
            if full_address:
                addresses_list.append(full_address)

        affiliations = ". ".join(addresses_list)

        # Subject categories
        subjects = get_nested(summary, "category_info", "subjects", "subject")
        if isinstance(subjects, list):
            subjects = "; ".join([s.get("content", "") for s in subjects])
        elif isinstance(subjects, dict):
            subjects = subjects.get("content", "")
        else:
            subjects = ""

        # ---- Build plain text record ----
        rec_text = (
            f"PT J\n"
            f"DT {dt}\n"
            f"AU {authors}\n"
            f"TI {title}\n"
            f"SO {source}\n"
            f"PY {year}\n"
            f"DI {doi}\n"
            f"DE {keywords}\n"
            f"ID {keywords_plus}\n"
            f"SC {subjects}\n"
            f"AB {abstract}\n"
            f"TC {tc_count}\n"
            f"NR {nr}\n"
            f"LA {language}\n"
            f"C1 {affiliations}\n"
            f"UT {uid}\n"
            f"ER\n\n"
        )
        records_text.append(rec_text)

# ---- Write output file ----
with open(output_file, "w", encoding="utf-8") as out:
    out.writelines(records_text)

print(f"✅ Extracted {len(records_text)} records to {output_file}")


## 2. Data cleaning

Remove articles published after 2025

In [ ]:
def split_wos_by_year(
    input_file="extracted_data.txt",
    keep_file="wos_up_to_2025.txt",
    removed_file="wos_2026.txt",
    excluded_years={"2026"}
):
    records = []
    current_record = []

    # --- Read and chunk the file into records ---
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if line.startswith("PT"):  # start of a new record
                current_record = [line]
            elif line.startswith("ER"):  # end of a record
                current_record.append(line)
                records.append(current_record)
                current_record = []
            else:
                if current_record is not None:
                    current_record.append(line)

    # --- Classify records ---
    kept = []
    removed = []

    for rec in records:
        year = None
        for line in rec:
            if line.startswith("PY "):  # publication year
                year = line.split()[1].strip()
                break

        if year in excluded_years:
            removed.append(rec)
        else:
            kept.append(rec)

    # --- Write results ---
    def write_records(filename, records):
        with open(filename, "w", encoding="utf-8") as f:
            for rec in records:
                for line in rec:
                    f.write(line + "\n")

    write_records(keep_file, kept)
    write_records(removed_file, removed)

    print(f"Done.\nKept: {len(kept)} records\nRemoved: {len(removed)} records")


# Run it
split_wos_by_year()


Remove articles with missing metadata (no title, no abstract)

In [ ]:
import os #import library to manipulate filenames

#start the function
def filter_articles_by_missing_fields(input_file):
    """
    Reads Web of Science records from a .txt file, where each article
    starts with a line beginnign with 'PT' and ends with a line beginning with 'ER'.
    
    Removes any article missing title (TI), abstract (AB), or authors (AU).
    Creates four output files:
      - *_valid.txt → all records that have all three fields
      - *_no_title.txt → missing title
      - *_no_abstract.txt → missing abstract
      - *_no_authors.txt → missing authors
    """
    
    base_name = os.path.splitext(input_file)[0] #splits the filename "extracted_wos_data_2.txt" into [0] extracted_wos_data_2 [1] .txt
    #define name of new txt files to be produced:
    valid_file = f"{base_name}_intact_info.txt"
    no_title_file = f"{base_name}_no_title.txt"
    no_abstract_file = f"{base_name}_no_abstract.txt"
    no_authors_file = f"{base_name}_no_authors.txt"

    #start the counts with 0
    total = 0
    valid_count = 0
    missing_title = 0
    missing_abstract = 0
    missing_authors = 0

    current_article = [] #this is used to temporarily store the article PT -> ER
    # Assume for now they do not have the information, if they do upon looking, we change to true
    has_title = False
    has_abstract = False
    has_authors = False

    #with opens all the files at once and when needed closes the file
    #r is read mode, and w is write mode
    #infile, out_valid, etc are our personal names to call the files
    with open(input_file, "r", encoding="utf-8") as infile, \
         open(valid_file, "w", encoding="utf-8") as out_valid, \
         open(no_title_file, "w", encoding="utf-8") as out_no_title, \
         open(no_abstract_file, "w", encoding="utf-8") as out_no_abstract, \
         open(no_authors_file, "w", encoding="utf-8") as out_no_authors:

        for line in infile: #reads each line in the infile file
            current_article.append(line) #appends each line to the current_article file

            # Check presence of key fields
            if line.startswith("TI ") and line.strip() != "TI": #if the line stsrts with "TI " and it's not (!=) an empty field after "TI " then change to true
                has_title = True
            elif line.startswith("AB ") and line.strip() != "AB" and "[No abstract available]" not in line:
                has_abstract = True
            elif line.startswith("AU ") and line.strip() != "AU":
                has_authors = True

            # End of one record
            if line.strip() == "ER":
                total += 1 #article has ended (we found the ER line) so add one to the total number of articles.

                # Categorize article based on missing fields

                #if title, abstract, and authors are there then put it in the out_valid file
                if has_title and has_abstract and has_authors:
                    out_valid.writelines(current_article)
                    valid_count += 1

                #if it is missing one then put it in one of these files (note an article can be added to more than one of these)
                else:
                    if not has_title:
                        out_no_title.writelines(current_article)
                        missing_title += 1
                    if not has_abstract:
                        out_no_abstract.writelines(current_article)
                        missing_abstract += 1
                    if not has_authors:
                        out_no_authors.writelines(current_article)
                        missing_authors += 1

                # Reset for next article
                current_article = []
                has_title = False
                has_abstract = False
                has_authors = False

    print(f"\nProcessed file: {input_file}")
    print(f"  Total articles: {total}")
    print(f"  Valid (with TI, AB, AU): {valid_count}")
    print(f"  Missing title: {missing_title}")
    print(f"  Missing abstract: {missing_abstract}")
    print(f"  Missing authors: {missing_authors}")
    
    if total > 0: #so we do not divide by 0 and get a crash
        print(f"  Retained {(valid_count/total):.1%} of records with complete metadata")

# --- Run the function with the input file we want ---
filter_articles_by_missing_fields("wos_up_to_2025.txt")


Remove non-English articles

In [ ]:
import os #import the os module to allow us to work with file paths and file names

#define the function
def filter_non_english_records(valid_file):
    """
    Reads a Web of Science text file of valid records (each starting with PT and ending with ER)
    and removes records that are not in English based on the 'LA' field.

    Produces two outputs:
      - *_valid_english.txt: records where LA contains 'English'
      - *_valid_non_english.txt: all other languages
    """
    
    base_name = os.path.splitext(valid_file)[0] #splits the file name into [0] extracted_wos_data_2 and [1] .txt
    english_file = f"{base_name}_english.txt"
    non_english_file = f"{base_name}_non_english.txt"

    #start the counts at 0
    total = 0
    english_count = 0
    non_english_count = 0

    current_article = [] #store the current article here temporarily
    language = None #assume the language is None for now

    #with allows us to close the files when we are done
    #open opens the file as read (r) or write (w) with personalised names eg infile.
    with open(valid_file, "r", encoding="utf-8") as infile, \
         open(english_file, "w", encoding="utf-8") as out_english, \
         open(non_english_file, "w", encoding="utf-8") as out_nonenglish:

        #loop through each line in the article metadata and append it to the current article up until we see the "ER" line and then we assume its the next article
        for line in infile:
            current_article.append(line)

            # Detect language field and what's after "LA "
            if line.startswith("LA "):
                language = line.strip()[3:]  # text after "LA " (3 characters L, A, space)
            
            if line.strip() == "ER":
                total += 1 #we have finished one article, so add a count to total

                # If the field tags exists AND language is English then appent it to out_english and add to count
                #language.lower makes it case-insensitive, also handle e.g. "English; Spanish" go to "english; spanish"
                if language and "english" in language.lower():
                    out_english.writelines(current_article)
                    english_count += 1

                #otherwise, add it to the out_nonenglish file
                else:
                    out_nonenglish.writelines(current_article)
                    non_english_count += 1

                # Reset for next record
                current_article = []
                language = None

    print(f"\nProcessed file: {valid_file}")
    print(f"  Total articles: {total}")
    print(f"  English articles: {english_count}")
    print(f"  Non-English articles: {non_english_count}")
    if total > 0:
        print(f"  Percentage English: {english_count/total:.1%}")

# --- Run it on your valid file ---
filter_non_english_records("wos_up_to_2025_intact_info.txt")


Remove articles published prior to 1993

In [ ]:
def split_wos_by_year(
    input_file="extracted_data.txt",
    keep_file="wos_up_to_2025.txt",
    removed_file="wos_excluded_years.txt",
    excluded_years={"2026"}
):
    records = []
    current_record = []

    # --- Read and chunk the file into records ---
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if line.startswith("PT"):  # start of a new record
                current_record = [line]
            elif line.startswith("ER"):  # end of a record
                current_record.append(line)
                records.append(current_record)
                current_record = []
            else:
                if current_record is not None:
                    current_record.append(line)

    # --- Classify records ---
    kept = []
    removed = []

    for rec in records:
        year = None
        for line in rec:
            if line.startswith("PY "):  # publication year
                year = line.split()[1].strip()
                break

        if year is not None and 1993 <= int(year) <= 2025:
            kept.append(rec)
        else:
            removed.append(rec)

    # --- Write results ---
    def write_records(filename, records):
        with open(filename, "w", encoding="utf-8") as f:
            for rec in records:
                for line in rec:
                    f.write(line + "\n")

    write_records(keep_file, kept)
    write_records(removed_file, removed)

    print(f"Done.\nKept: {len(kept)} records\nRemoved: {len(removed)} records")


# Run it
split_wos_by_year()

## 3. Coarse-grained LLM classifiaction to categorise articles into biology or not biology based on the following definitions:
- Biology = Research on living organisms or biological processes.
- Not Biology = Research primarily on physical systems, engineering, modeling, social sciences, or environmental processes without biological mechanisms.

In [ ]:
import datetime
from ollama import chat
import os
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- Model ----
MODEL = "qwen3:8b"

SYSTEM_PROMPT = """
You are an expert scientific document classifier. Determine whether a scientific abstract is primarily about **Biology** or **Not Biology**.

Definitions:
- Biology = Research on living organisms or biological processes.
- Not Biology = Research primarily on physical systems, engineering, modeling, social sciences, or environmental processes without biological mechanisms.

Instructions:
1. Only classify abstracts based on their primary scientific focus.
2. Output ONLY JSON in this format:

[
  {"id": 1, "Category": "Biology"},
  {"id": 2, "Category": "Not Biology"}
]

Do NOT add commentary or explanations.
"""

# ---- Step 1: Parse abstracts from WoS txt file ----
def extract_abstracts_from_wos(input_file):
    abstracts = []
    with open(input_file, "r", encoding="utf-8") as f:
        current_lines = []
        recording = False
        for line in f:
            line = line.rstrip()
            if line.startswith("PT J"):
                current_lines = [line]
                recording = True
                continue
            if recording:
                current_lines.append(line)
                if line.startswith("ER"):
                    abstracts.append(current_lines)
                    recording = False
    return abstracts

# ---- Step 2: Classify a batch of abstracts ----
def classify_batch(batch):
    """
    batch = list of tuples: (index, abstract_text)
    """
    user_prompt = "\n".join([f'{{"id": {idx+1}, "abstract": "{txt}"}}' for idx, txt in batch])
    try:
        response = chat(
            model=MODEL,
            options={"temperature": 0, "seed": 42, "enable_thinking": False}, #thinking false means we do not see the chain of thought, only the final answer
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ]
        )
        content = response["message"]["content"].strip()
        # Parse JSON-like response
        matches = re.findall(r'"id":\s*(\d+),\s*"Category":\s*"(.+?)"', content)
        result = {int(idx): cat for idx, cat in matches}
        return result
    except Exception as e:
        # fallback: mark all in batch as Not Biology
        return {idx+1: "Not Biology" for idx, _ in batch}

# ---- Step 3: Main processing ----
def main(input_file, output_file, biology_file, not_biology_file, batch_size=30, max_workers=4):
    abstracts = extract_abstracts_from_wos(input_file)
    print(f"Found {len(abstracts)} abstracts to classify.")
    
    start_time = datetime.datetime.now()
    
    # Prepare abstract texts
    abstract_texts = [(i, " ".join([l[3:].strip() for l in lines if l.startswith("AB ")]))
                      for i, lines in enumerate(abstracts)]
    
    # Split into batches
    batches = [abstract_texts[i:i+batch_size] for i in range(0, len(abstract_texts), batch_size)]
    
    results = {}
    
    # Parallel processing
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(classify_batch, batch): batch for batch in batches}
        for future in as_completed(futures):
            batch_result = future.result()
            results.update(batch_result)
    
    # ---- Write updated files ----
    os.makedirs(os.path.dirname(output_file), exist_ok=True) if os.path.dirname(output_file) else None
    
    with open(output_file, "w", encoding="utf-8") as fout, \
         open(biology_file, "a", encoding="utf-8") as bio_fout, \
         open(not_biology_file, "a", encoding="utf-8") as notbio_fout:
        
        for i, lines in enumerate(abstracts, start=1):
            category = results.get(i, "Not Biology")
            kp_value = "1" if category == "Biology" else "0"
            
            new_lines = []
            for line in lines:
                if line.startswith("ER"):
                    new_lines.append(f"KP {kp_value}")
                new_lines.append(line)
            
            fout.write("\n".join(new_lines) + "\n\n")
            
            # Append to biology / not_biology
            if kp_value == "1":
                bio_fout.write(" ".join(new_lines) + "\n\n")
            else:
                notbio_fout.write(" ".join(new_lines) + "\n\n")
            
            elapsed = datetime.datetime.now() - start_time
            print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Abstract {i}/{len(abstracts)}: KP {kp_value} ({category}) (elapsed: {elapsed})")
    
    print("✅ Classification complete.")

# ---- Run ----
if __name__ == "__main__":
    main(
        input_file="wos_abm.txt",
        output_file="classified.txt",
        biology_file="biology.txt",
        not_biology_file="not_biology.txt",
        batch_size=10,
        max_workers=4
    )


## 4. Fine-grained LLM classification to categorise articles as oncology, epidemiology, ecology, or other-biology based on the following definitions:

- oncology:
   Use ONLY if cancer or oncology is the central research subject.
   Cancer must be explicitly studied (e.g. tumors, carcinogenesis, cancer treatment, cancer-specific therapies).
   Do NOT use if cancer is only mentioned as motivation or comparison.

- epidemiology:
   Use if the PRIMARY focus is disease or health outcomes analyzed across human or animal populations, i.e., epidemiology.
   Do NOT use if disease is a minor component of ecological or environmental dynamics.

- ecology:
   Use if the study focuses on how living organisms interact with each other and their environment, i.e., ecology.
   Includes wildlife disease ONLY when ecological dynamics (not public health) are the primary outcome.

- other:
   Use if the study does NOT clearly fit oncology, epidemiology, or ecology.
   Includes all other biology subfields, review papers, or methods papers without a clear focus on health-related event in a population, ecosystem, or cancer.

In [ ]:
import datetime #to print out the times for output (elapsed time for each abstract)
from ollama import chat #allows us to send prompts to the LLM and return responses
import os #allows us to create output directories
import re #regex, to extract field tags like KA
from concurrent.futures import ThreadPoolExecutor, as_completed #lets us run multiple tasks at the same time, ThreadPoolExecutor runs multiple LLM calls at once, as_completed processes the results when they finish
import time #not used in current code
import random #not used in current code

# Model (from ollama)
MODEL = "qwen3:8b"

#system prompt:
SYSTEM_PROMPT_KA = """
You are an expert biological sciences classifier.

Classify each abstract into exactly ONE PRIMARY category.

You MUST determine the PRIMARY scientific focus of the study, not secondary mentions or background context.

--------------------------------------------------
DECISION ORDER (follow strictly):

1. oncology
   Use ONLY if cancer or oncology is the central research subject.
   Cancer must be explicitly studied (e.g. tumors, carcinogenesis, cancer treatment, cancer-specific therapies).
   Do NOT use if cancer is only mentioned as motivation or comparison.

2. epidemiology
   Use if the PRIMARY focus is disease or health outcomes analyzed across human or animal populations, i.e., epidemiology.
   Do NOT use if disease is a minor component of ecological or environmental dynamics.

3. ecology
   Use if the study focuses on how living organisms interact with each other and their environment, i.e., ecology.
   Includes wildlife disease ONLY when ecological dynamics (not public health) are the primary outcome.

4. other
   Use if the study does NOT clearly fit oncology, epidemiology, or ecology.
   Includes all other biology subfields, review papers, or methods papers without a clear focus on health-related event in a population, ecosystem, or cancer.

--------------------------------------------------

Rules:
- Choose the MOST SPECIFIC applicable category.
- Do NOT infer intent beyond what is stated.
- Never guess.
- Respond ONLY in valid JSON as a list of objects:
  [{"id": 1, "KA": "oncology"}, ...]
"""

### Step 1: Parse abstracts from WoS txt file
# creates list, and records every line starting from PT J and ending with ER.
def extract_abstracts_from_wos(input_file): #define fucntion
    abstracts = [] #create list -- each element of the list will be one article metadata
    with open(input_file, "r", encoding="utf-8") as f: #opens file in "r" read mode using UTF-8 encoding (importnat for special characters), "with" ensures the file is automatically closed
        current_lines = [] #temporary list of current article
        recording = False #assume we are currently not within an article (PT J -> ER)
        for line in f: #reads file one line at a time
            line = line.rstrip() #removes the \n
            if line.startswith("PT J"): #check if the line starts with PT J
                current_lines = [line] #if yes, then start a new article block
                recording = True #start recording the article metadata
                continue #skip to next if statement, we have already found PT J
            if recording: #if recording is true (it is if we are in an article)
                current_lines.append(line) #add every libe to the current article
                if line.startswith("ER"): #do so until we hit the "ER" to signify end of article
                    abstracts.append(current_lines) #save entire block
                    recording = False #exit article mode and start looking for next PT J line
    return abstracts #returns what we recorded eg ["PT J", "AU Smith J", "AB ...", "ER"]

### Step 2: Classify a batch of abstracts
def classify_batch_ka(batch):
    #loop over each batch where each batch contains idx = the index of the abstract in the batch and txt = the text from the abstract
    #create output variable: {"id": 1, "abstract": "Animal movement is central to ecology..."}
    #the f denotes an f string so result as follows: {"id": 1, "abstract": "This is the text of the abstract."} - a json like string
    #"\n".join means to this and make a string of one abstract per line: ["a", "b", "c"] -> "\n".join(["a","b","c"]) -> "a\nb\nc"
    user_prompt = "\n".join(
        [f'{{"id": {idx+1}, "abstract": "{txt}"}}' for idx, txt in batch]
    )

    try: #attempt to call the LLM (chat) and process the output, if anything goes wrong - go to "except" 
        response = chat( #call the chat function from the ollama API to classify abstracts
            model=MODEL, #use our chosen model
            options={"temperature": 0, "seed": 42, "enable_thinking": False},  #control LLM behaviour: temp=0 for deterministic output, seed=42 for repeats, enable_thinking=False to suppress the chain of thought 
            messages=[ #this is what we send to the LLM
                {"role": "system", "content": SYSTEM_PROMPT_KA}, #system instructions - tells the LLM what to do exactly via the prompt
                {"role": "user", "content": user_prompt} #user message containing the batch of abstracts to classify - the list of abstracts 
            ]
        )

        content = response["message"]["content"].strip() #retreieve the actual text output of the LLM, response is a dictionary returned by chat in the path ["message"]["content"], .strip() removes any extra whitespace at the start or end (like new line)

        #returns LLM results as: [("1", "oncology"), ("2", "ecology")] via re.findall(pattern, text)- ie find al matches of pattern inside text and return them as a list.
        #r means raw string, so take backslashes literally (not like \n means new line).
        #"id":\s* --> Search the text in content, find every place that looks like "id": 3, where \s is any white space (space, tab, new line), * zero or more times - so it can capture "id":1, "id: 1"
        #(\d+): \d means a digit (0-9), + means one or more, () means capture this 
        #find now eg "KA": "ecology" and extract the id number and the KA category.
        # "(.+?)" captures the category string. capture between ( and ), . means any character, + means one of more, ? means stop as soon as possible
        matches = re.findall(r'"id":\s*(\d+),\s*"KA":\s*"(.+?)"', content)
        return {
            # results dictionary converts matches into a dictionary with abstracts as integers (int(idx)) and values as categories (ka.lower() makes it lowercase)
            #eg {"results": {1: "oncology", 2: "ecology"}, "count": 2}
            "results": {int(idx): ka.lower() for idx, ka in matches},
            "count": len(batch) #counts the number of abstracts in this batch
        }

    except Exception: #if any error occurs in the try block then come here
        # fallback: mark all as "other"
        return {
            "results": {idx+1: "other" for idx, _ in batch},
            "count": len(batch)
        }

# Step 3: Main processing 
def main(input_file, output_file, batch_size, max_workers):
    abstracts = extract_abstracts_from_wos(input_file) #return list of articles - each line like: ["PT J", "AU ...", "AB ...", "ER"]
    print(f"Found {len(abstracts)} biology abstracts to sub-classify.")

    start_time = datetime.datetime.now() #store current time to use for "elapsed time" later...

    #extract only the abstacts (AB) eg [(0, "this study..."), (1, "this one...")]
    abstract_texts = [] #creates empty list to store abstract text
    #for each article number (i) and lines (list of lines for that article):
    for i, lines in enumerate(abstracts): #enumerate(abstracts) turns [PT J \n AB ...] to (0, abstract[0])
        abstract_text = " ".join( #put a space between each string and put them together, eg ["hello", "world"] → "hello world"
            [l[3:].strip() for l in lines if l.startswith("AB ")] #for each line, only look at lines that start with "AB ", then remove first 3 characters and extra space at start or end
        )
        abstract_texts.append((i, abstract_text)) #make list (0,abstract), (1, abstract)

    
    #split abstracts into batches for the LLM call
    batches = [
        abstract_texts[i:i+batch_size]
        for i in range(0, len(abstract_texts), batch_size)
    ]

    results = {} #maps abstract_id to KA
    completed = 0 #counts how many abstracts are done so far

    #runs multiple batches simulteneously (each batch = one LLM call)
    with ThreadPoolExecutor(max_workers=max_workers) as executor: #each thread can run one LLM call, with makes sure things are cleaned up and no dangling processes when done
        futures = {
            executor.submit(classify_batch_ka, batch): batch #Run classify_batch_ka(batch) in another thread, and start it immediately, and return a future (a promise that we will see later)
            for batch in batches
        }

        # iterate over futures as they complete, collect the results (in order they finish not sent)
        for future in as_completed(futures): #Give me futures one at a time, as soon as they finish. as one finishes it becomes future
            result = future.result() #retrieves the return value of classify_batch_ka(batch)
            results.update(result["results"]) #Add all key-value pairs from this batch into the global results dictionary.
            completed += result["count"] #increase "completed" count by "result["count"]" (completed = completed + result["count"])

            elapsed = datetime.datetime.now() - start_time #get elapsed time to print
            
            print(
                f"[{datetime.datetime.now().strftime('%H:%M:%S')}] "
                f"✅ Batch finished ({result['count']} abstracts) — "
                f"total processed: {completed}/{len(abstracts)} "
                f"(elapsed: {elapsed})"
            )

    #make sure the output folder exists before writing the fille
    #IF the output file has a directory path, create it, IF NOT  (i.e. its just output.txt then do nothing
    #os.path.dirname(output_file) extracts the folder path from the output filename eg if results/output.txt then return "results", if output.txt then return ""
    #os.makedirs(path, exist_ok=True) creates the directory (and parent directories if needed)
    #exist_ok=True means dont crash if the folder already exists
    os.makedirs(os.path.dirname(output_file), exist_ok=True) if os.path.dirname(output_file) else None

    with open(output_file, "w", encoding="utf-8") as fout: #open output file in write (w) mode using utf-8 to keep special characters, as fout, with means automatically close file when done
        for i, lines in enumerate(abstracts, start=1): #for every article (1,2,3) and list of lines in each article, enumerate makes each article 1 ["PT J", "AU JOhn"] instead of on new lines
            ka_value = results.get(i, "other") #look up the KA value, if it doesnt exist use "other"

            new_lines = [] #rebuild article"
            for line in lines: #for each line in the article
                if line.startswith("ER"): #place the new KA "category" just before the ER.
                    new_lines.append(f"KA {ka_value}")
                new_lines.append(line) #always add the original lines unchanged

            fout.write("\n".join(new_lines) + "\n\n") #write to fout the article to the output file with new line between each, and add 2 extra newlines after each article!

            elapsed = datetime.datetime.now() - start_time #print total runtime so far
            print(
                f"[{datetime.datetime.now().strftime('%H:%M:%S')}] "
                f"Abstract {i}/{len(abstracts)}: KA {ka_value} (elapsed: {elapsed})"
            )

    print("✅ KA sub-classification complete.")

# ---- Run ----
if __name__ == "__main__":
    main(
        input_file="biology.txt",
        output_file="classifications.txt",
        batch_size=5,
        max_workers=4
    )
